<a href="https://colab.research.google.com/github/iasdf31578/anesthesia-checklist/blob/main/%E6%8E%92%E7%8F%AD%E5%B0%8F%E5%B9%AB%E6%89%8B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
#!/usr/bin/env python3
"""
每日人力分配排班程式 v2（Colab / Jupyter 版）

【每月執行前，只需修改下方「設定區」即可，不需要改其他地方】
"""

import pandas as pd

# ============================================================
#  【設定區】每月執行前填好這裡
# ============================================================

# ① 本月人員身分
#    A - 刀房內，主要院區台北
#    B - 刀房內，主要院區淡水
#    C - 特殊科，主要院區台北
#    D - 特殊科，主要院區淡水
DOCTOR_TYPE = {
    "k": "A",
    "n": "A",
    "m": "B",
    "r": "B",
    "t": "B",
    "j": "C",
    "o": "C",
    "s": "C",
    "u": "C",
    # 範例：D 類人員可在此加入，例如 "x": "D"
}

# ② 上個月最後一天的值班人員（本月第一天 OFF 的人）
#    多人請用逗號分隔，例如 "k,n"；無人填空字串 ""
PREV_TAIPEI_ONCALL  = "o"   # 上月最後一天台北值班
PREV_DANSHUI_ONCALL = "j"   # 上月最後一天淡水值班

# ③ 檔案路徑
INPUT_FILE  = "202604.csv"
OUTPUT_FILE = "202604_output.csv"
ENCODING    = "big5"   # 若出現編碼錯誤請改為 "big5" or "utf-8-sig"

# ============================================================
#  以下不需要修改
# ============================================================

WEEKDAY_COL = "Unnamed: 1"
WEEKDAYS    = {"一","二","三","四","五","1","2","3","4","5"}  # 中文或數字皆支援


def parse_cell(val) -> list:
    if pd.isna(val) or str(val).strip() in ("", "nan"):
        return []
    return [x.strip() for x in str(val).split(",") if x.strip()]


def parse_setting(raw: str) -> list:
    return [x.strip() for x in raw.split(",") if x.strip()] if raw.strip() else []


def fmt(people: list) -> str:
    return "".join(people)


def fmt_bracket(people: list) -> str:
    return "(" + "".join(people) + ")" if people else ""


def allocate_ab(remaining: list) -> tuple:
    pool_a = [d for d in remaining if DOCTOR_TYPE.get(d) == "A"]
    pool_b = [d for d in remaining if DOCTOR_TYPE.get(d) == "B"]
    tp, ds = [], []
    while pool_a or pool_b:
        if pool_a:   tp.append(pool_a.pop(0))
        elif pool_b: tp.append(pool_b.pop(0))
        else: break
        if pool_b:   ds.append(pool_b.pop(0))
        elif pool_a: ds.append(pool_a.pop(0))
        else: break
    return tp, ds


def process_schedule(df, prev_taipei_on, prev_danshui_on):
    all_doctors = list(DOCTOR_TYPE.keys())

    for col in ["台北白班", "淡水白班", "OFF"]:
        if col not in df.columns:
            df[col] = ""
        else:
            df[col] = df[col].fillna("").astype(str)

    for idx, row in df.iterrows():
        taipei_on   = parse_cell(row.get("台北值班"))
        danshui_on  = parse_cell(row.get("淡水值班"))
        taipei_opd  = parse_cell(row.get("台北門診"))
        danshui_opd = parse_cell(row.get("淡水門診"))
        leave       = parse_cell(row.get("假"))

        # 判斷是否工作日
        weekday = str(row.get(WEEKDAY_COL, "")).strip().rstrip(".0")  # 處理整數/浮點/中文
        is_weekday = weekday in WEEKDAYS
        SATURDAY = {"六", "6"}

        if not is_weekday:
            # 星期六：輸出 OFF（前一日值班者），但不排白班
            if weekday in SATURDAY:
                off_list = list(dict.fromkeys(prev_taipei_on + prev_danshui_on))
                df.at[idx, "OFF"] = fmt(off_list)
                # 星期六寫完 OFF 後清空 prev，避免星期一重複使用
                prev_taipei_on  = []
                prev_danshui_on = []
            # 若六日有值班，更新 prev 供下個工作日使用
            if taipei_on or danshui_on:
                prev_taipei_on  = taipei_on[:]
                prev_danshui_on = danshui_on[:]
            continue

        # ① OFF
        off_list = list(dict.fromkeys(prev_taipei_on + prev_danshui_on))

        # ② 排除名單
        exclude = (set(off_list) | set(taipei_on) | set(danshui_on)
                   | set(taipei_opd) | set(danshui_opd) | set(leave))

        # ③ 值班者入主體
        tp_main = [p for p in taipei_on  if DOCTOR_TYPE.get(p) in ("A", "B", "D")]
        ds_main = [p for p in danshui_on if DOCTOR_TYPE.get(p) in ("A", "B", "C")]

        # ④ 值班者例外入括號
        tp_brk  = [p for p in taipei_on  if DOCTOR_TYPE.get(p) == "C"]
        ds_brk  = [p for p in danshui_on if DOCTOR_TYPE.get(p) == "D"]

        # ⑤ 剩餘 A/B 交替補入
        remaining_ab = [d for d in all_doctors
                        if DOCTOR_TYPE.get(d) in ("A", "B") and d not in exclude]
        tp_add, ds_add = allocate_ab(remaining_ab)
        tp_main += tp_add
        ds_main += ds_add

        # ⑤-補 台北主體仍少於淡水，從 ds_add 中移一人（優先A）到台北
        if len(tp_main) < len(ds_main) and ds_add:
            candidate = next((p for p in ds_add if DOCTOR_TYPE.get(p) == "A"), None)
            if candidate is None:
                candidate = next((p for p in ds_add if DOCTOR_TYPE.get(p) == "B"), None)
            if candidate:
                ds_main.remove(candidate)
                tp_main.append(candidate)

        # ⑥ 剩餘 C/D 入括號
        tp_brk += [d for d in all_doctors if DOCTOR_TYPE.get(d) == "C" and d not in exclude]
        ds_brk += [d for d in all_doctors if DOCTOR_TYPE.get(d) == "D" and d not in exclude]

        # ⑥-補 若括號外沒有人，將當天該院區值班者從括號移出到主體
        #   台北、淡水各自獨立判斷；只移值班者（括號內的其他剩餘C/D不動）
        if not tp_main:
            oncall_in_brk = [p for p in taipei_on if p in tp_brk]
            if oncall_in_brk:
                pick = oncall_in_brk[0]
                tp_brk.remove(pick)
                tp_main.append(pick)

        if not ds_main:
            oncall_in_brk = [p for p in danshui_on if p in ds_brk]
            if oncall_in_brk:
                pick = oncall_in_brk[0]
                ds_brk.remove(pick)
                ds_main.append(pick)

        # ⑦ 寫入
        df.at[idx, "台北白班"] = fmt(tp_main)  + fmt_bracket(tp_brk)
        df.at[idx, "淡水白班"] = fmt(ds_main) + fmt_bracket(ds_brk)
        df.at[idx, "OFF"]     = fmt(off_list)

        prev_taipei_on  = taipei_on[:]
        prev_danshui_on = danshui_on[:]

    return df


# ── 執行 ──────────────────────────────────────────
df = pd.read_csv(INPUT_FILE, encoding=ENCODING)

prev_tp = parse_setting(PREV_TAIPEI_ONCALL)
prev_ds = parse_setting(PREV_DANSHUI_ONCALL)

print("人員身分設定：")
for code in ["A", "B", "C", "D"]:
    members = [k for k, v in DOCTOR_TYPE.items() if v == code]
    print(f"  {code}：{members or '（無）'}")
print(f"上月末台北值班：{prev_tp or '（無）'}")
print(f"上月末淡水值班：{prev_ds or '（無）'}")

df = process_schedule(df, prev_tp, prev_ds)

df.to_csv(OUTPUT_FILE, index=False, encoding=ENCODING)
print(f"\n排班完成！輸出檔案：{OUTPUT_FILE}")

cols = ["日期", WEEKDAY_COL, "台北值班", "淡水值班",
        "台北門診", "淡水門診", "台北白班", "淡水白班", "OFF", "假"]
cols = [c for c in cols if c in df.columns]
print("\n─── 結果預覽 ───")
print(df[cols].to_string(index=False))

人員身分設定：
  A：['k', 'n']
  B：['m', 'r', 't']
  C：['j', 'o', 's', 'u']
  D：（無）
上月末台北值班：['o']
上月末淡水值班：['j']

排班完成！輸出檔案：202604_output.csv

─── 結果預覽 ───
 日期 Unnamed: 1 台北值班 淡水值班 台北門診 淡水門診      台北白班 淡水白班 OFF     假
  1          三    k    t    u    j    knr(s)   tm  oj     o
  2          四    o    r  NaN  NaN   nm(osu)    r  kt     j
  3          五  NaN  NaN  NaN  NaN   kn(jsu)   mt  or   NaN
  4          六  NaN  NaN  NaN  NaN                      NaN
  5          日  NaN  NaN  NaN  NaN                      NaN
  6          一  NaN  NaN  NaN  NaN knt(josu)   mr       NaN
  7          二    o    r  NaN    j   nt(osu)    r       k,m
  8          三    j    t  NaN  NaN   nm(jsu)    t  or     k
  9          四    u    m  NaN  NaN   nr(uos)    m  jt     k
 10          五    s    n  NaN  NaN    kr(sj)    n  um   o,t
 11          六  NaN  NaN  NaN  NaN                 sn     t
 12          日  NaN  NaN  NaN  NaN                        t
 13          一    n    k  NaN  NaN  nm(josu)   kr         t
 14          